# Reality AI — Production LoRA Fine-Tuning (Mistral-7B)

This notebook fine-tunes **Mistral-7B-v0.1** using QLoRA (4-bit quantised base
model + LoRA adapters) on the Reality AI property-chat dataset.

**Environment:** Kaggle GPU (T4 16 GB or P100 16 GB)  
**Dataset:** `full_dataset.jsonl` (~750 synthetic conversations)  
**Output:** LoRA adapter weights (zipped for download)  

The adapter teaches the model to behave as "RealityAI" — a Surat real-estate
chat assistant that understands property types, pricing in INR, Surat
neighbourhoods, and amenity-aware search.

### How this fits in the project
- **Person 1 (AI track)** trains and exports the adapter here.
- The adapter is downloaded and placed at `ai/finetune/adapters/reality-ai-lora/`.
- `ai/inference/` loads the adapter at runtime to power the chat endpoint.
- See `ai/docs/ADR-001-base-model.md` for the model selection rationale.

> **Kaggle GPU quota:** Free tier = 30 hrs/week. This run takes ~1-2 hrs on T4.
> Monitor wall-clock time in Cell 7 output.

In [ ]:
# ============================================================================
# Cell 2: Install dependencies
# ============================================================================
# Pin versions close to what was tested locally to avoid API drift.
# bitsandbytes provides 4-bit quantisation; trl provides SFTTrainer.

!pip install -q \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=1.0.0 \
    datasets>=2.20.0 \
    trl>=0.12.0

In [ ]:
# ============================================================================
# Cell 3: Imports and configuration constants
# ============================================================================

import os
import time
import shutil
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# ── Configuration ──────────────────────────────────────────────────────────
# Change MODEL_NAME to test a different base model.
# DATASET_PATH assumes full_dataset.jsonl is uploaded as a Kaggle Dataset
# and attached to this notebook (edit the path if your dataset name differs).

MODEL_NAME: str = "mistralai/Mistral-7B-v0.1"

# Kaggle dataset path — adjust the dataset slug if yours is different.
# When you add the dataset in Kaggle, it typically appears at:
#   /kaggle/input/<dataset-slug>/full_dataset.jsonl
DATASET_PATH: str = "/kaggle/input/reality-ai-finetune-data/full_dataset.jsonl"

OUTPUT_DIR: str = "/kaggle/working/reality-ai-lora"
ZIP_OUTPUT: str = "/kaggle/working/reality-ai-lora"  # .zip appended by shutil

# Training hyperparameters
NUM_EPOCHS: int = 3
LEARNING_RATE: float = 2e-4
PER_DEVICE_BATCH_SIZE: int = 2      # T4 16GB can handle batch=2 comfortably
GRADIENT_ACCUMULATION_STEPS: int = 4  # effective batch = 8
MAX_SEQ_LENGTH: int = 2048

# ── Sanity checks ─────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU detected — enable GPU in Kaggle settings!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH}")
print(f"Effective batch size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

In [ ]:
# ============================================================================
# Cell 4: Load and preprocess the dataset
# ============================================================================
# Each row in the JSONL is {"messages": [{"role": ..., "content": ...}, ...]}.
# We apply the tokenizer's chat template to convert messages into the format
# the base model expects, producing a single "text" column for SFTTrainer.

print(f"Loading dataset from: {DATASET_PATH}")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Dataset size: {len(dataset)} examples")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

def apply_chat_template(example):
    """Format the messages list using the tokenizer's chat template."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template, remove_columns=dataset.column_names)
print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}")
print(f"\n... ({len(dataset[0]['text'])} chars total)")

In [ ]:
# ============================================================================
# Cell 5: Load the base model in 4-bit quantisation
# ============================================================================
# Even on a 16GB T4, 4-bit quantisation is worthwhile:
#   - Mistral-7B at fp16 = ~14 GB (barely fits, no room for training)
#   - Mistral-7B at 4-bit = ~5 GB (leaves ~11 GB for optimizer/activations)
#
# NF4 (Normal Float 4) with double quantisation is the standard QLoRA recipe.

# Detect compute dtype — T4 is Turing (fp16 only), P100 is Pascal (fp16 only)
# Ampere+ GPUs (A100, etc.) support bfloat16.
if torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    print("Using bfloat16 compute dtype.")
else:
    compute_dtype = torch.float16
    print("Using float16 compute dtype (GPU does not support bf16).")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"\nLoading {MODEL_NAME} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # incompatible with gradient checkpointing
print("Model loaded.")

In [ ]:
# ============================================================================
# Cell 6: Apply LoRA adapters
# ============================================================================
# These values are inlined from ai/finetune/configs/lora_config_full.yaml.
# If you change the YAML, update these to match.
#
# Key differences from the debug config (lora_config.yaml):
#   - r=16 (vs 8) — more capacity for the larger dataset
#   - dropout=0.05 (vs 0.10) — 750 examples reduces overfitting risk
#   - target_modules: q/k/v/o_proj (Mistral's separate attention layers)
#     vs qkv_proj/o_proj (Phi-3's fused attention)

lora_config = LoraConfig(
    # r (rank): 16 gives ~1% trainable params — good balance between
    # capacity and VRAM on 16GB. Double the debug config's r=8.
    r=16,

    # lora_alpha: 2x rank convention (32/16 = 2.0 effective scaling).
    # Keeps the update magnitude consistent with the debug config.
    lora_alpha=32,

    # lora_dropout: 5% — lighter than debug (10%) since we have 7.5x
    # more training examples (750 vs 100).
    lora_dropout=0.05,

    # target_modules: Mistral-7B uses SEPARATE Q/K/V projections
    # (not fused like Phi-3). These names must match the model's
    # actual nn.Linear layer names.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],

    # task_type: autoregressive language modelling for chat fine-tuning.
    task_type="CAUSAL_LM",

    bias="none",
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
pct = 100.0 * trainable / total
print(f"Trainable parameters: {trainable:,} / {total:,} ({pct:.2f}%)")

In [ ]:
# ============================================================================
# Cell 7: Configure and run training
# ============================================================================
# We use SFTConfig (trl 0.15+) which extends TrainingArguments with
# SFT-specific fields like max_seq_length and packing.
#
# Wall-clock timing is printed every 10 steps so you can estimate total
# runtime and track usage against Kaggle's 30-hour weekly GPU quota.

use_bf16 = (compute_dtype == torch.bfloat16)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    logging_steps=10,         # print loss every 10 steps
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    dataset_text_field="text",
    report_to="none",          # no wandb on Kaggle by default
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

# ── Train with wall-clock tracking ────────────────────────────────────────
print("=" * 60)
print(f"Starting training: {NUM_EPOCHS} epochs, "
      f"effective batch={PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Total steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else '(calculated at start)'}")
print("=" * 60)

t0 = time.time()
result = trainer.train()
elapsed = time.time() - t0

# ── Report ────────────────────────────────────────────────────────────────
mins, secs = divmod(int(elapsed), 60)
hrs, mins = divmod(mins, 60)
time_str = f"{hrs}h {mins}m {secs}s" if hrs else f"{mins}m {secs}s"

print("\n" + "=" * 60)
print(f"  Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Wall-clock:     {time_str}")
print(f"  Trainable:      {trainable:,} params ({pct:.2f}%)")
print(f"  GPU hours used: {elapsed / 3600:.2f} / 30.0 weekly quota")
print("=" * 60)

In [ ]:
# ============================================================================
# Cell 8: Save adapter and create downloadable zip
# ============================================================================
# save_pretrained() saves ONLY the LoRA adapter weights (a few MB),
# not the full 7B model. The adapter can be loaded later with:
#   peft.AutoPeftModelForCausalLM.from_pretrained(path)

print(f"Saving adapter to {OUTPUT_DIR} ...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List saved files
saved_files = list(Path(OUTPUT_DIR).rglob("*"))
total_size_mb = sum(f.stat().st_size for f in saved_files if f.is_file()) / 1024**2
print(f"Adapter files: {len(saved_files)} files, {total_size_mb:.1f} MB total")
for f in sorted(saved_files):
    if f.is_file():
        print(f"  {f.relative_to(OUTPUT_DIR)}  ({f.stat().st_size / 1024:.0f} KB)")

# Create a single .zip for easy download from Kaggle's Output tab
zip_path = shutil.make_archive(ZIP_OUTPUT, "zip", root_dir=OUTPUT_DIR)
zip_size_mb = Path(zip_path).stat().st_size / 1024**2
print(f"\nZipped adapter: {zip_path} ({zip_size_mb:.1f} MB)")
print("Download this from the Output tab on the right. -->")

## Next Steps

### 1. Download the adapter
- Click the **Output** tab in the right sidebar of this Kaggle notebook.
- Download `reality-ai-lora.zip`.

### 2. Place it in your local repo
```
ai/finetune/adapters/reality-ai-lora/
    adapter_config.json
    adapter_model.safetensors
    tokenizer_config.json
    tokenizer.json
    special_tokens_map.json
    ...
```
Unzip the downloaded file into `ai/finetune/adapters/reality-ai-lora/`.

### 3. Load it in inference
The `ai/inference/` module will load this adapter at runtime:
```python
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained(
    "ai/finetune/adapters/reality-ai-lora/",
    device_map="auto",
    torch_dtype=torch.float16,
)
```

### 4. Don't commit the adapter to git
The `ai/finetune/adapters/` directory is already in `.gitignore`.
Store the zip in Google Drive or Kaggle Datasets for team access.

> **Re-training:** If you update the dataset, just re-run this notebook.
> The full run takes ~1-2 hours on a T4 GPU.